## Исследование обученной модели Word2Vec

В данном ноутбуке исследуются свойства обученной модели Word2Vec. Рассматриваются основные возможности библиотеки Gensim для работы с векторными представлениями слов, включая поиск похожих слов, вычисление семантической близости, выполнение арифметических операций над векторами слов и решение аналогий.

### План работы

1. Загрузка обученной модели.
2. Поиск наиболее похожих слов.
3. Вычисление семантической близости слов.
4. Арифметические операции над векторами.
5. Решение аналогий.
6. Визуализация векторных представлений слов.
7. Работа с векторами слов.
8. Выводы.

In [ ]:
from gensim.models import Word2Vec

In [ ]:
# загружаем модель skip-gram 8e
model = Word2Vec.load("../models/skipgram_100d_8e.model")

In [ ]:
# поиск похожих слов 
model.wv.most_similar("король")

In [ ]:
# семантическая близость
model.wv.similarity("чай", "кофе")

In [ ]:
# не похожие слова
model.wv.doesnt_match([
    "яблоко",
    "груша",
    "банан",
    "автомобиль"
])

In [ ]:
# арифметика векторов
model.wv.most_similar(
    positive=["король", "женщина"],
    negative=["мужчина"]
)

In [ ]:
# расстояние между словами
model.wv.distance(
    "кошка",
    "собака"
)

In [ ]:
# работа с векторами
vector = model.wv["кошка"]

vector.shape

In [ ]:
# первые координаты
vector[:10]

In [ ]:
# какие слова отсутсвуют
"чатгпт" in model.wv

In [ ]:
# самые частые слова
model.wv.index_to_key[:30]

### Визуализация векторных представлений слов

Для наглядного анализа структуры пространства эмбеддингов выполнена визуализация векторных представлений слов. Поскольку размерность векторов равна 100, для отображения на плоскости использованы методы снижения размерности PCA и t-SNE.

В качестве примера выбраны слова, относящиеся к нескольким семантическим группам.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [ ]:
groups = {
    "Животные": [
        "кошка", "собака", "тигр", "лев", "волк"
    ],
    "Транспорт": [
        "машина", "автобус", "поезд", "самолет", "корабль"
    ],
    "Фрукты": [
        "яблоко", "банан", "апельсин", "груша", "лимон"
    ],
    "Столицы": [
        "москва", "париж", "берлин", "рим", "мадрид"
    ],
    "Страны": [
        "россия", "франция", "германия", "италия", "испания"
    ],
}

In [ ]:
# проверяем что все слова есть в словаре

words = []
vectors = []
labels = []

for group, group_words in groups.items():
    for word in group_words:
        if word in model.wv:
            words.append(word)
            vectors.append(model.wv[word])
            labels.append(group)

vectors = np.array(vectors)

### Визуализация с использованием PCA

In [ ]:
pca = PCA(n_components=2)

vectors_2d = pca.fit_transform(vectors)

In [ ]:
plt.figure(figsize=(12, 10))

for group in groups.keys():
    indices = [
        i for i, label in enumerate(labels)
        if label == group
    ]

    plt.scatter(
        vectors_2d[indices, 0],
        vectors_2d[indices, 1],
        label=group,
        s=60,
    )

for i, word in enumerate(words):
    plt.text(
        vectors_2d[i, 0],
        vectors_2d[i, 1],
        word,
        fontsize=9,
    )

plt.title("PCA")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Визуализация с использованием t-SNE

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=5,
    random_state=42,
)

vectors_tsne = tsne.fit_transform(vectors)

In [ ]:
plt.figure(figsize=(12, 10))

for group in groups.keys():
    indices = [
        i for i, label in enumerate(labels)
        if label == group
    ]

    plt.scatter(
        vectors_tsne[indices, 0],
        vectors_tsne[indices, 1],
        label=group,
        s=60,
    )

for i, word in enumerate(words):
    plt.text(
        vectors_tsne[i, 0],
        vectors_tsne[i, 1],
        word,
        fontsize=9,
    )

plt.title("t-SNE")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Выводы
Обученная модель формирует осмысленное пространство эмбеддингов. Слова одной семантической группы располагаются ближе друг к другу, чем слова, относящиеся к различным предметным областям.